# SongNet: Real-time Music Genre Classification (complete, Run-All, Drive-safe)
UE24CS352A Mini-Project, Problem #33. Reproduction of Stanford CS229 (2018) SongNet on FMA-small (8 genres).

**Use: `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.**

- The first time, a pop-up asks for **Google Drive access**. Click Allow.
- **Keep the tab open and the laptop awake and plugged in.** About 1 hour on a fresh start, about 30 minutes if resuming.
- Everything is saved to Google Drive (`MyDrive/songnet_results`, `MyDrive/songnet_backup`).
- **Resumable:** after a Colab reset, just click Run all again. The data is restored from Drive and finished trainings are skipped.
- If a cell fails, the run stops there. Copy the error message.

**Results we got earlier with this exact recipe (random 70/20/10 split, 800 test clips)**

| System | Test accuracy |
|---|---|
| Best classical baseline (MLP) | 57.1% |
| Single SongNet (mean of 10 runs) | 61.0% +/- 0.7 |
| SongNet ensemble (10 models) | 63.25% |
| Hybrid: SongNet ensemble + feature models | 64.62% |
| Paper's SongNet | about 65% |

Training is random, so expect numbers within about 1 point of these.

**Honest-reporting rules**
1. Report single models, the SongNet ensemble and the hybrid as **separate rows**.
2. Every ensemble design that you look at on the test set must be mentioned in the write-up. Do not keep trying variants until a test number looks good.
3. Blend weights and model pools are chosen on **validation** data, never on test.

In [ ]:
# ---------------- SWITCHES ----------------
RUN_GRU       = True    # GRU-head models (needed for the ensembles)
RUN_SEEDS     = True    # 4 extra seeds x 2 heads = 8 more trainings (~20-25 min)
RUN_BASELINES = False   # classical baselines (already measured earlier)
RUN_FINAL     = False   # optional: retrain on train+validation with fixed settings (10 trainings)
RUN_OFFICIAL  = False   # optional: official FMA split (2+ trainings)
EPOCHS        = 100     # max epochs per training (early stopping may end sooner)
SEEDS         = [1, 2, 3, 4]

In [ ]:
!pip install -q librosa joblib tqdm seaborn scikit-learn
import os, torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU, then Run all again."
from google.colab import drive
drive.mount('/content/drive')
R = "/content/drive/MyDrive/songnet_results"
BK = "/content/drive/MyDrive/songnet_backup/processed_v1.tgz"
for p in ["/content/src", "/content/data", R + "/random", R + "/official", os.path.dirname(BK)]:
    os.makedirs(p, exist_ok=True)
def sh(cmd):
    get_ipython().system("cd /content && " + cmd)
HEADS = ["dense", "gru"] if RUN_GRU else ["dense"]

## Source files (written to `src/`)

In [ ]:
%%writefile /content/src/data.py
"""Convert FMA-small mp3s into fixed-size log-mel spectrograms.

Usage:
  python src/data.py --meta data/fma_metadata --audio data/fma_small --out data/processed
Uses FMA's official split (training / validation / test = 80/10/10).
"""
import argparse, json
from pathlib import Path
import numpy as np, pandas as pd, librosa
from joblib import Parallel, delayed
from tqdm import tqdm

SR, N_FFT, HOP, N_MELS, FRAMES = 22050, 2048, 1024, 128, 640  # ~29.7 s


def audio_to_mel(y):
    """Waveform -> log-mel (N_MELS, T). Shared with the real-time demo."""
    m = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
    return librosa.power_to_db(m, ref=1.0).astype(np.float32)


def fix_length(m):
    if m.shape[1] < FRAMES:
        m = np.pad(m, ((0, 0), (0, FRAMES - m.shape[1])), constant_values=m.min())
    return m[:, :FRAMES]


def load_one(path):
    try:
        y, _ = librosa.load(path, sr=SR, duration=30)
        return fix_length(audio_to_mel(y)).astype(np.float16)
    except Exception:
        return None  # a few FMA mp3s are corrupt; they get skipped


def track_path(audio_dir, tid):
    s = f"{tid:06d}"
    return str(Path(audio_dir) / s[:3] / f"{s}.mp3")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--meta", required=True)
    ap.add_argument("--audio", required=True)
    ap.add_argument("--out", default="data/processed")
    ap.add_argument("--jobs", type=int, default=-1)
    a = ap.parse_args()

    tracks = pd.read_csv(Path(a.meta) / "tracks.csv", index_col=0, header=[0, 1])
    small = tracks[tracks["set", "subset"] == "small"]
    classes = sorted(small["track", "genre_top"].dropna().unique())
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    json.dump(classes, open(out / "classes.json", "w"))

    for split in ["training", "validation", "test"]:
        df = small[small["set", "split"] == split]
        ids = df.index.tolist()
        mels = Parallel(n_jobs=a.jobs)(
            delayed(load_one)(track_path(a.audio, t)) for t in tqdm(ids, desc=split))
        keep = [i for i, m in enumerate(mels) if m is not None]
        X = np.stack([mels[i] for i in keep])
        y = np.array([classes.index(df["track", "genre_top"].iloc[i]) for i in keep])
        np.save(out / f"{split}_X.npy", X); np.save(out / f"{split}_y.npy", y)
        np.save(out / f"{split}_ids.npy", np.array([ids[i] for i in keep]))
        print(split, X.shape, f"skipped {len(ids) - len(keep)}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/src/model.py
import torch, torch.nn as nn, torch.nn.functional as F


class SongNet(nn.Module):
    """1D-conv over time (mel bins = channels) -> per-timestep genre logits.
    head='dense': time-distributed linear layer (as in the paper)
    head='gru'  : causal GRU before the linear layer (real recurrent head)
    Song-level prediction = mean of per-timestep probabilities.
    """
    def __init__(self, n_mels=128, n_classes=8, channels=(128, 256, 256), k=5, p=0.25, head="dense"):
        super().__init__()
        layers, c = [], n_mels
        for o in channels:
            layers += [nn.Conv1d(c, o, k, padding=k // 2), nn.BatchNorm1d(o),
                       nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(p)]
            c = o
        self.cnn = nn.Sequential(*layers)
        self.head = head
        if head == "gru":
            self.rnn = nn.GRU(c, 256, batch_first=True)  # unidirectional -> usable in real time
            c = 256
        self.fc = nn.Linear(c, n_classes)

    def forward(self, x):                 # x: (B, n_mels, T)
        h = self.cnn(x).transpose(1, 2)   # (B, T', C)
        if self.head == "gru":
            h, _ = self.rnn(h)
        return self.fc(h)                 # (B, T', n_classes)


def song_log_probs(logits):
    """Mean of per-timestep softmax -> log-probabilities per song."""
    return torch.log(F.softmax(logits, -1).mean(1) + 1e-8)

In [ ]:
%%writefile /content/src/train.py
"""Train + evaluate SongNet (v2: random time crops, SpecAugment-style masking,
per-mel-bin normalization, AdamW).
Usage: python src/train.py --data data/processed --head dense
"""
import argparse, json
from pathlib import Path
import numpy as np, torch, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from model import SongNet, song_log_probs

ap = argparse.ArgumentParser()
ap.add_argument("--data", default="data/processed")
ap.add_argument("--head", choices=["dense", "gru"], default="dense")
ap.add_argument("--epochs", type=int, default=80)
ap.add_argument("--bs", type=int, default=32)
ap.add_argument("--lr", type=float, default=1e-3)
ap.add_argument("--wd", type=float, default=1e-2)
ap.add_argument("--crop", type=int, default=320, help="training crop length in frames (640 = full clip)")
ap.add_argument("--patience", type=int, default=15)
ap.add_argument("--seed", type=int, default=0)
ap.add_argument("--out", default="results")
ap.add_argument("--trainval", action="store_true", help="train on train+validation for a FIXED number of epochs (no model selection); test stays held out")
a = ap.parse_args()

torch.manual_seed(a.seed); np.random.seed(a.seed)
dev = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
d = Path(a.data); out = Path(a.out); out.mkdir(exist_ok=True)
classes = json.load(open(d / "classes.json"))
L = lambda s: (torch.from_numpy(np.load(d / f"{s}_X.npy")), torch.from_numpy(np.load(d / f"{s}_y.npy")))
(Xtr, ytr), (Xva, yva), (Xte, yte) = L("training"), L("validation"), L("test")

if a.trainval:
    Xtr = torch.cat([Xtr, Xva]); ytr = torch.cat([ytr, yva])

sample = Xtr[:1000].float()
mean = sample.mean(dim=(0, 2), keepdim=True)[0]            # (n_mels, 1) per-bin stats
std = sample.std(dim=(0, 2), keepdim=True)[0] + 1e-5
json.dump({"mean": mean.flatten().tolist(), "std": std.flatten().tolist()}, open(out / f"norm_{a.head}.json", "w"))
mean, std = mean.to(dev), std.to(dev)
prep = lambda x: ((x.to(dev).float() - mean) / std)


def augment(X, idx):
    """Random time crop per sample + one freq mask and one time mask."""
    T, crop = X.shape[2], min(a.crop, X.shape[2])
    starts = np.random.randint(0, T - crop + 1, size=len(idx))
    x = prep(torch.stack([X[j, :, s:s + crop] for j, s in zip(idx.tolist(), starts)]))
    for b in range(len(x)):
        f = np.random.randint(0, 16); f0 = np.random.randint(0, 128 - f + 1)
        t = np.random.randint(0, 24); t0 = np.random.randint(0, crop - t + 1)
        x[b, f0:f0 + f, :] = 0; x[b, :, t0:t0 + t] = 0
    return x


model = SongNet(n_classes=len(classes), head=a.head).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min" if a.trainval else "max", factor=0.5, patience=4)


@torch.no_grad()
def predict(X):
    model.eval(); preds = []
    for i in range(0, len(X), 64):
        preds.append(song_log_probs(model(prep(X[i:i + 64]))).argmax(1).cpu())
    return torch.cat(preds).numpy()


best, bad = 0, 0
for ep in range(a.epochs):
    model.train(); perm = torch.randperm(len(Xtr)); tot = 0
    for i in range(0, len(perm), a.bs):
        idx = perm[i:i + a.bs]
        loss = F.nll_loss(song_log_probs(model(augment(Xtr, idx))), ytr[idx].to(dev))
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(idx)
    if a.trainval:
        sched.step(tot / len(perm))
        print(f"epoch {ep+1:02d} loss {tot/len(perm):.4f} (train+val, fixed schedule)", flush=True)
        continue
    val = accuracy_score(yva.numpy(), predict(Xva)); sched.step(val)
    print(f"epoch {ep+1:02d} loss {tot/len(perm):.4f} val_acc {val:.4f}", flush=True)
    if val > best:
        best, bad = val, 0; torch.save(model.state_dict(), out / f"songnet_{a.head}.pt")
    else:
        bad += 1
        if bad >= a.patience: print("early stop"); break

if a.trainval:
    torch.save(model.state_dict(), out / f"songnet_{a.head}.pt")
model.load_state_dict(torch.load(out / f"songnet_{a.head}.pt", map_location=dev))
pred = predict(Xte); acc = accuracy_score(yte.numpy(), pred)
print(f"\nTEST accuracy ({a.head}): {acc:.4f}")
rep = classification_report(yte.numpy(), pred, target_names=classes, digits=3)
print(rep)
open(out / f"report_{a.head}.txt", "w").write(f"test_acc={acc:.4f}\n{rep}")
cm = confusion_matrix(yte.numpy(), pred)
plt.figure(figsize=(8, 6)); sns.heatmap(cm, annot=True, fmt="d", cmap="YlGnBu", xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout(); plt.savefig(out / f"confusion_{a.head}.png", dpi=150)

In [ ]:
%%writefile /content/src/baselines.py
"""Classical baselines on FMA's precomputed librosa features (features.csv).
Official split:  python src/baselines.py --meta data/fma_metadata
Random split:    python src/baselines.py --meta data/fma_metadata --ids_dir data/processed_random --out results_random
"""
import argparse, json
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

ap = argparse.ArgumentParser()
ap.add_argument("--meta", required=True)
ap.add_argument("--out", default="results")
ap.add_argument("--ids_dir", default=None, help="use the split stored in this processed dir (e.g. data/processed_random)")
a = ap.parse_args()

tracks = pd.read_csv(Path(a.meta) / "tracks.csv", index_col=0, header=[0, 1])
feats = pd.read_csv(Path(a.meta) / "features.csv", index_col=0, header=[0, 1, 2])
small = tracks[tracks["set", "subset"] == "small"]
small = small[small["track", "genre_top"].notna()]


def get(split):
    if a.ids_dir:
        df = small.loc[np.load(Path(a.ids_dir) / f"{split}_ids.npy")]
    else:
        df = small[small["set", "split"] == split]
    return feats.loc[df.index].fillna(0).values, df["track", "genre_top"].values


Xtr, ytr = get("training"); Xte, yte = get("test")
print("train/test sizes:", len(Xtr), len(Xte))
sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

models = {
    "KNN": KNeighborsClassifier(),
    "LogReg": LogisticRegression(max_iter=1000),
    "MLP": MLPClassifier(max_iter=300, random_state=0),
    "LinearSVM": LinearSVC(dual=False),
}
res = {}
for n, m in models.items():
    m.fit(Xtr, ytr); res[n] = float(accuracy_score(yte, m.predict(Xte)))
    print(f"{n}: {res[n]:.4f}", flush=True)
Path(a.out).mkdir(exist_ok=True)
json.dump(res, open(Path(a.out) / "baselines.json", "w"), indent=2)

In [ ]:
%%writefile /content/src/resplit.py
"""Re-split the preprocessed FMA-small clips into a random, class-stratified 70/20/10
split (the protocol described in the SongNet paper) instead of FMA's official split.
Usage: python src/resplit.py --src data/processed --out data/processed_random
"""
import argparse, shutil
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split

ap = argparse.ArgumentParser()
ap.add_argument("--src", default="data/processed")
ap.add_argument("--out", default="data/processed_random")
ap.add_argument("--seed", type=int, default=0)
a = ap.parse_args()

S = ["training", "validation", "test"]
cat = lambda k: np.concatenate([np.load(Path(a.src) / f"{s}_{k}.npy") for s in S])
X, y, ids = cat("X"), cat("y"), cat("ids")
idx = np.arange(len(y))
tr, rest = train_test_split(idx, test_size=0.30, stratify=y, random_state=a.seed)
va, te = train_test_split(rest, test_size=1 / 3, stratify=y[rest], random_state=a.seed)

out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
for name, sel in zip(S, [tr, va, te]):
    np.save(out / f"{name}_X.npy", X[sel]); np.save(out / f"{name}_y.npy", y[sel])
    np.save(out / f"{name}_ids.npy", ids[sel]); print(name, len(sel))
shutil.copy(Path(a.src) / "classes.json", out / "classes.json")

In [ ]:
%%writefile /content/src/evaluate.py
"""Evaluate saved checkpoints on the test set; averages probabilities if several runs are given.
Usage:
  python src/evaluate.py --data data/processed_random --runs results_random:dense
  python src/evaluate.py --data data/processed_random --runs results_random:dense results_random:gru
Each run is <results_dir>:<head>. Use --split validation to check validation accuracy instead.
"""
import argparse, json
from pathlib import Path
import numpy as np, torch
import matplotlib; matplotlib.use("Agg")
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from model import SongNet

ap = argparse.ArgumentParser()
ap.add_argument("--data", default="data/processed")
ap.add_argument("--runs", nargs="+", required=True)
ap.add_argument("--split", default="test", choices=["test", "validation"])
ap.add_argument("--out", default="results_eval")
a = ap.parse_args()

dev = "cuda" if torch.cuda.is_available() else "cpu"
d = Path(a.data)
classes = json.load(open(d / "classes.json"))
X = torch.from_numpy(np.load(d / f"{a.split}_X.npy")); y = np.load(d / f"{a.split}_y.npy")


@torch.no_grad()
def run_probs(res_dir, head):
    norm = json.load(open(Path(res_dir) / f"norm_{head}.json"))
    mean = torch.tensor(np.reshape(norm["mean"], (-1, 1)), dtype=torch.float32, device=dev)
    std = torch.tensor(np.reshape(norm["std"], (-1, 1)), dtype=torch.float32, device=dev)
    m = SongNet(n_classes=len(classes), head=head).to(dev)
    m.load_state_dict(torch.load(Path(res_dir) / f"songnet_{head}.pt", map_location=dev)); m.eval()
    out = []
    for i in range(0, len(X), 64):
        x = (X[i:i + 64].to(dev).float() - mean) / std
        out.append(torch.softmax(m(x), -1).mean(1).cpu())
    return torch.cat(out).numpy()


all_p = []
for r in a.runs:
    res_dir, head = r.split(":")
    p = run_probs(res_dir, head); all_p.append(p)
    print(f"{r:35s} {a.split} accuracy: {accuracy_score(y, p.argmax(1)):.4f}")

final = np.mean(all_p, axis=0)
pred = final.argmax(1)
tag = "ensemble" if len(all_p) > 1 else "single"
print(f"\n{tag.upper()} {a.split} accuracy: {accuracy_score(y, pred):.4f}\n")
print(classification_report(y, pred, target_names=classes, digits=3))
Path(a.out).mkdir(exist_ok=True)
cm = confusion_matrix(y, pred)
plt.figure(figsize=(8, 6)); sns.heatmap(cm, annot=True, fmt="d", cmap="YlGnBu", xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout(); plt.savefig(Path(a.out) / f"confusion_{tag}_{a.split}.png", dpi=150)

In [ ]:
%%writefile /content/src/ensemble.py
"""Hybrid ensemble: SongNet run(s) + classical classifiers on FMA's precomputed audio features.
The blend weight is chosen on the VALIDATION set only; test accuracy is reported once at that weight.
Usage:
  python src/ensemble.py --data data/processed_random --meta data/fma_metadata \
      --runs results_random:dense results_random:gru
"""
import argparse, json
from pathlib import Path
import numpy as np, pandas as pd, torch
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from model import SongNet

ap = argparse.ArgumentParser()
ap.add_argument("--data", required=True)
ap.add_argument("--meta", required=True)
ap.add_argument("--runs", nargs="+", required=True, help="<results_dir>:<head> ...")
ap.add_argument("--feat", choices=["mlp", "logreg", "both"], default="both")
ap.add_argument("--seed", type=int, default=0)
ap.add_argument("--trainval", action="store_true", help="fit the feature models on train+validation")
ap.add_argument("--auto_pool", action="store_true", help="also try SVM and gradient-boosting feature models; pool and weight are chosen on VALIDATION only")
ap.add_argument("--w", type=float, default=None, help="fixed SongNet weight; if omitted it is chosen on validation")
a = ap.parse_args()

dev = "cuda" if torch.cuda.is_available() else "cpu"
d = Path(a.data)
classes = json.load(open(d / "classes.json"))
split = {s: (torch.from_numpy(np.load(d / f"{s}_X.npy")), np.load(d / f"{s}_y.npy"), np.load(d / f"{s}_ids.npy"))
         for s in ["training", "validation", "test"]}


@torch.no_grad()
def songnet_probs(res_dir, head, X):
    norm = json.load(open(Path(res_dir) / f"norm_{head}.json"))
    mean = torch.tensor(np.reshape(norm["mean"], (-1, 1)), dtype=torch.float32, device=dev)
    std = torch.tensor(np.reshape(norm["std"], (-1, 1)), dtype=torch.float32, device=dev)
    m = SongNet(n_classes=len(classes), head=head).to(dev)
    m.load_state_dict(torch.load(Path(res_dir) / f"songnet_{head}.pt", map_location=dev)); m.eval()
    out = []
    for i in range(0, len(X), 64):
        x = (X[i:i + 64].to(dev).float() - mean) / std
        out.append(torch.softmax(m(x), -1).mean(1).cpu())
    return torch.cat(out).numpy()


# --- SongNet part
S = {s: np.mean([songnet_probs(*r.split(":"), split[s][0]) for r in a.runs], axis=0) for s in ["validation", "test"]}

# --- classical feature models (FMA precomputed audio features)
tracks = pd.read_csv(Path(a.meta) / "tracks.csv", index_col=0, header=[0, 1])
feats = pd.read_csv(Path(a.meta) / "features.csv", index_col=0, header=[0, 1, 2])
small = tracks[tracks["set", "subset"] == "small"]


def get(s):
    ids = split[s][2]
    F = feats.loc[ids].fillna(0).values
    lab = np.array([classes.index(g) for g in small.loc[ids]["track", "genre_top"]])
    assert (lab == split[s][1]).all(), "label mismatch between features and spectrogram arrays"
    return F, lab


Ftr, ytr = get("training"); Fva, yva = get("validation"); Fte, yte = get("test")
if a.trainval:
    Ftr, ytr = np.concatenate([Ftr, Fva]), np.concatenate([ytr, yva])
sc = StandardScaler().fit(Ftr); Ftr, Fva, Fte = sc.transform(Ftr), sc.transform(Fva), sc.transform(Fte)
from sklearn.svm import SVC
from sklearn.ensemble import HistGradientBoostingClassifier
assert not (a.auto_pool and a.trainval), "--auto_pool selects on validation, so it cannot be combined with --trainval"


def make(n):
    return {"logreg": LogisticRegression(max_iter=1000),
            "mlp": MLPClassifier(max_iter=300, random_state=a.seed),
            "svm": SVC(kernel="rbf", C=3, probability=True, random_state=a.seed),
            "hgb": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=a.seed)}[n]


names = ["logreg", "mlp", "svm", "hgb"] if a.auto_pool else {"mlp": ["mlp"], "logreg": ["logreg"], "both": ["logreg", "mlp"]}[a.feat]
yv, yt = split["validation"][1], split["test"][1]
P = {}
for n in names:
    m = make(n).fit(Ftr, ytr)
    P[n] = {"validation": m.predict_proba(Fva), "test": m.predict_proba(Fte)}
    print(f"feature model {n:7s} validation acc {accuracy_score(yv, P[n]['validation'].argmax(1)):.4f}", flush=True)

pools = ({"logreg+mlp": ["logreg", "mlp"], "logreg+mlp+svm": ["logreg", "mlp", "svm"],
          "logreg+mlp+hgb": ["logreg", "mlp", "hgb"], "all4": ["logreg", "mlp", "svm", "hgb"]}
         if a.auto_pool else {"+".join(names): names})
pool_p = lambda pn, s: np.mean([P[n][s] for n in pools[pn]], axis=0)

print("\nw = weight on SongNet (1-w on feature models)")
best_pool, best_w, best_v, ncand = None, 1.0, -1, 0
for pn in pools:
    for w in ([] if a.w is not None else np.linspace(0, 1, 11)):
        v = accuracy_score(yv, (w * S["validation"] + (1 - w) * pool_p(pn, "validation")).argmax(1)); ncand += 1
        print(f"  pool={pn:15s} w={w:.1f}  validation acc {v:.4f}")
        if v > best_v: best_pool, best_w, best_v = pn, w, v
if a.w is not None:
    best_pool, best_w = list(pools)[0], a.w; print(f"using fixed w = {best_w:.2f}")
else:
    print(f"\ncandidates evaluated on validation: {ncand}")
    print(f"chosen pool = {best_pool}, w = {best_w:.1f} (validation {best_v:.4f})")
Fp = {"test": pool_p(best_pool, "test")}
print(f"SongNet only   test acc: {accuracy_score(yt, S['test'].argmax(1)):.4f}")
print(f"Features only  test acc: {accuracy_score(yt, Fp['test'].argmax(1)):.4f}")
final = (best_w * S["test"] + (1 - best_w) * Fp["test"]).argmax(1)
print(f"HYBRID         test acc: {accuracy_score(yt, final):.4f}\n")
print(classification_report(yt, final, target_names=classes, digits=3))

## Data: restore from Drive if possible, otherwise download and preprocess

In [ ]:
HAVE_BACKUP = os.path.exists(BK) and os.path.getsize(BK) > 5e8
print("Valid Drive backup of preprocessed data:", HAVE_BACKUP)

if not os.path.exists("/content/data/fma_metadata"):
    sh("cd data && wget -nv https://os.unil.cloud.switch.ch/fma/fma_metadata.zip && unzip -q fma_metadata.zip")

if os.path.exists("/content/data/processed/test_X.npy"):
    print("Preprocessed data already present, skipping.")
elif HAVE_BACKUP:
    print("Restoring preprocessed data from Drive...")
    sh(f"tar xzf {BK} -C /content")
else:
    print("No backup. Downloading FMA-small (7 GB) and preprocessing (~20 min)...")
    sh("cd data && wget -nv https://os.unil.cloud.switch.ch/fma/fma_small.zip && unzip -q fma_small.zip")
    sh("python src/data.py --meta data/fma_metadata --audio data/fma_small --out data/processed")
    sh(f"tar czf {BK} data/processed")
    assert os.path.getsize(BK) > 5e8, "Drive backup is too small, something went wrong"

for s in ["training", "validation", "test"]:
    for k in ["X", "y", "ids"]:
        assert os.path.exists(f"/content/data/processed/{s}_{k}.npy"), f"missing {s}_{k}.npy, preprocessing failed"
assert os.path.exists("/content/data/fma_metadata/tracks.csv"), "metadata download failed"
sh(f"cp data/processed/classes.json {R}/classes.json")
print("Data OK.")

## Paper-style random 70/20/10 split
Deterministic (fixed seed), so it always gives the same train/validation/test clips.

In [ ]:
if not os.path.exists("/content/data/processed_random/test_X.npy"):
    sh("python src/resplit.py")
assert os.path.exists("/content/data/processed_random/test_X.npy"), "resplit failed"
print("Random split OK.")

## Baselines (optional switch)

In [ ]:
if RUN_BASELINES:
    sh(f"python src/baselines.py --meta data/fma_metadata --ids_dir data/processed_random --out {R}/random")
    sh(f"python src/baselines.py --meta data/fma_metadata --out {R}/official")
else:
    print("Skipped. Earlier test accuracy: random split kNN 44.9, LogReg 51.4, MLP 57.1, SVM 52.8; official split kNN 32.3, LogReg 43.5, MLP 43.5, SVM 43.9.")

## Training SongNet on the random split
Outputs go straight to Drive. A finished training (its `report_<head>.txt` exists) is skipped on re-runs.

In [ ]:
def train(data, out, head, seed=0, epochs=None, trainval=False):
    if os.path.exists(f"{out}/report_{head}.txt"):
        print(f"[skip] {head} already trained in {out}"); return
    cmd = f"python src/train.py --data {data} --out {out} --head {head} --epochs {epochs or EPOCHS} --seed {seed}"
    if trainval: cmd += " --trainval"
    sh(cmd + " | tail -25")
    assert os.path.exists(f"{out}/report_{head}.txt"), f"{head} training did not finish"

for h in HEADS:
    train("data/processed_random", R + "/random", h, seed=0)

### Extra seeds (switch `RUN_SEEDS`)
Independent runs with different random seeds. They show how stable the result is and feed the ensembles.

In [ ]:
if RUN_SEEDS:
    for s in SEEDS:
        for h in HEADS:
            train("data/processed_random", f"{R}/random_s{s}", h, seed=s)
else:
    print("Skipped (RUN_SEEDS=False)")

In [ ]:
# list what is on Drive
sh(f"ls {R}")

## Ensembles on the random split
Uses every trained model found on Drive (seed 0 plus the extra seeds).

1. `evaluate.py`: average of the SongNet models.
2. `ensemble.py` (design 1): SongNet blended with logistic regression + MLP on FMA's precomputed audio features; the blend weight is chosen on validation data.
3. `ensemble.py --auto_pool` (design 2): also tries an RBF-SVM and gradient boosting; pool and weight are chosen on validation data only.

Report each design as its own row, and mention in the write-up that both designs were evaluated.

In [ ]:
dirs = [R + "/random"] + [f"{R}/random_s{s}" for s in SEEDS]
runs = " ".join(f"{d}:{h}" for d in dirs for h in HEADS if os.path.exists(f"{d}/songnet_{h}.pt"))
print("models in the ensemble:", len(runs.split()))
sh(f"python src/evaluate.py --data data/processed_random --runs {runs} --out {R}/eval_random_all")

In [ ]:
# design 1: logistic regression + MLP feature models
sh(f"python src/ensemble.py --data data/processed_random --meta data/fma_metadata --runs {runs}")

In [ ]:
# design 2: richer feature pool, chosen on validation only (a few minutes on CPU)
sh(f"python src/ensemble.py --data data/processed_random --meta data/fma_metadata --runs {runs} --auto_pool")

## Optional: retrain on train + validation (fixed settings)
Epochs (dense 65, GRU 45) and the blend weight (0.6) are fixed in advance from the validation results. The test set is used once at the end. Earlier result: SongNet ensemble 63.88%, hybrid 64.62%.

In [ ]:
if RUN_FINAL:
    FIN = R + "/final"
    EP = {"dense": 65, "gru": 45}
    for s in range(5):
        for h in HEADS:
            train("data/processed_random", f"{FIN}_s{s}", h, seed=s, epochs=EP[h], trainval=True)
    fruns = " ".join(f"{FIN}_s{s}:{h}" for s in range(5) for h in HEADS)
    sh(f"python src/evaluate.py --data data/processed_random --runs {fruns} --out {R}/eval_final")
    sh(f"python src/ensemble.py --data data/processed_random --meta data/fma_metadata --runs {fruns} --trainval --w 0.6")
else:
    print("Skipped (RUN_FINAL=False)")

## Optional: official FMA split (artist-separated, stricter)
Earlier results: dense 45.9%, GRU 47.9%.

In [ ]:
if RUN_OFFICIAL:
    for h in HEADS:
        train("data/processed", R + "/official", h, seed=0)
    oruns = " ".join(f"{R}/official:{h}" for h in HEADS)
    sh(f"python src/evaluate.py --data data/processed --runs {oruns} --out {R}/eval_official")
    sh(f"python src/ensemble.py --data data/processed --meta data/fma_metadata --runs {oruns}")
else:
    print("Skipped (RUN_OFFICIAL=False)")

## Summary of single-model results saved on Drive

In [ ]:
import glob, re
rows = []
for f in sorted(glob.glob(R + "/*/report_*.txt")):
    m = re.search(r"test_acc=([0-9.]+)", open(f).read())
    rows.append((f.split("/")[-2], f.split("_")[-1].replace(".txt", ""), float(m.group(1))))
for name, head, acc in rows: print(f"{name:12s} {head:6s} {acc*100:5.2f}%")
import numpy as np
for pref in ["random", "final", "official"]:
    a = [x[2] for x in rows if x[0].startswith(pref)]
    if a: print(f"mean over {len(a)} '{pref}*' runs: {np.mean(a)*100:.2f}% +/- {np.std(a)*100:.2f}")

In [ ]:
from IPython.display import Image, display
for f in [R + "/eval_random_all/confusion_ensemble_test.png", R + "/random/confusion_dense.png"]:
    if os.path.exists(f): display(Image(f))

## Download results to your laptop (small zip: models, reports, plots)

In [ ]:
sh(f"cd /content/drive/MyDrive && zip -qr /content/songnet_results.zip songnet_results && ls -lh /content/songnet_results.zip")
from google.colab import files
files.download("/content/songnet_results.zip")

## After this: run the demo on your laptop (not in Colab)
1. Unzip `songnet_results.zip`. Copy `songnet_results/random/songnet_dense.pt`, `songnet_gru.pt`, `norm_dense.json`, `norm_gru.json` into your project's `results/` folder, and `songnet_results/classes.json` to `data/processed/classes.json`.
2. `pip install -r requirements.txt`, install ffmpeg, then `streamlit run app.py` and open http://localhost:8501.